In [ ]:
import os
import cognee

GAMMAPY_CTA_HGPS_DATASET = "gammapy-cta-hgps"
HESS_PAPERS_DATASET = "hess-papers"
DATASET_NAMES = [GAMMAPY_CTA_HGPS_DATASET, HESS_PAPERS_DATASET]

In [ ]:
# reset system
# await cognee.forget(everything=True)

In [ ]:
import json
import shutil
import subprocess
import urllib.request

RESOURCES_FILE = "../data/resources.json"

with open(RESOURCES_FILE, "r") as f:
    resources = json.load(f)

# download resources
PAPER_DIR = "../data/cache/ingestion/related_papers"
os.makedirs(PAPER_DIR, exist_ok=True)

for paper in resources["related_papers"]:
    arxiv_id = paper["url"].rsplit("/", 1)[-1]
    paper["local_path"] = os.path.join(PAPER_DIR, f"{arxiv_id}.pdf")

    if not os.path.exists(paper["local_path"]):
        print(f"Downloading {paper["url"]}")
        urllib.request.urlretrieve(paper["url"], paper["local_path"])

WHITEPAPER_DIR = "../data/cache/ingestion/whitepaper"
os.makedirs(WHITEPAPER_DIR, exist_ok=True)

whitepaper = resources["whitepaper"]
whitepaper_id = whitepaper["url"].rsplit("/", 1)[-1]
whitepaper["local_path"] = os.path.join(WHITEPAPER_DIR, f"{whitepaper_id}.pdf")
if not os.path.exists(whitepaper["local_path"]):
    print(f"Downloading {whitepaper["url"]}")
    urllib.request.urlretrieve(whitepaper["url"], whitepaper["local_path"])

# one sparse clone of gammapy repository for both documentation and examples
# url like https://github.com/{owner}/{repo}/tree/{ref}/{folder_path}
REPO_GROUPS = ["documentation", "examples"]
_, _, _, owner, repo, _, ref, *_ = resources["documentation"][0]["url"].split("/")
repo_url = f"https://github.com/{owner}/{repo}.git"

items_with_paths = []
for group in REPO_GROUPS:
    group_dir = f"../data/cache/ingestion/{group}"
    for item in resources[group]:
        folder_path = "/".join(item["url"].split("/")[7:])
        item["local_path"] = os.path.join(group_dir, os.path.basename(folder_path))
        if not os.path.exists(item["local_path"]):
            items_with_paths.append((item, group_dir, folder_path))

if items_with_paths:
    print(f"Cloning {repo_url}")
    tmp_dir = "../data/gammapy_repo.tmp"
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", ref, "--filter=blob:none", "--sparse",
         repo_url, tmp_dir],
        check=True,
    )
    subprocess.run(
        ["git", "-C", tmp_dir, "sparse-checkout", "set", *(fp for _, _, fp in items_with_paths)],
        check=True,
    )

    for group in REPO_GROUPS:
        os.makedirs(f"../data/cache/ingestion/{group}", exist_ok=True)
    for item, _, folder_path in items_with_paths:
        shutil.move(os.path.join(tmp_dir, folder_path), item["local_path"])
    shutil.rmtree(tmp_dir)

In [ ]:
gammapy_paper = [p for p in resources["related_papers"] if p["url"].endswith("2308.13584")]
hgps_paper = [p for p in resources["related_papers"] if p["url"].endswith("1804.02432")]
related_papers = [
    p for p in resources["related_papers"]
    if p not in gammapy_paper and p not in hgps_paper
]
# related papers are sorted by citation count
related_papers.sort(key=lambda p: (p["n_citations"] is None, -(p["n_citations"] or 0)))

ingestion_items = {
    GAMMAPY_CTA_HGPS_DATASET: (
        resources["documentation"]
        + resources["examples"]
        + gammapy_paper
        + hgps_paper
        + [resources["whitepaper"]]
    ),
    HESS_PAPERS_DATASET: related_papers,
}
for items in ingestion_items.values():
    for item in items:
        item["ingested"] = False

In [ ]:
import math
import shutil

from pypdf import PdfReader, PdfWriter
from tqdm import tqdm

# split PDFs into parts to avoid ingestion crash
TMP_SPLIT_DIR = "../data/cache/ingestion/tmp_split"
os.makedirs(TMP_SPLIT_DIR, exist_ok=True)
PAGES_PER_PART = 10


def split_pdf(item):
    reader = PdfReader(item["local_path"])
    n_pages = len(reader.pages)
    if n_pages <= PAGES_PER_PART:
        return [item["local_path"]]

    base = os.path.splitext(os.path.basename(item["local_path"]))[0]
    part_dir = os.path.join(TMP_SPLIT_DIR, base)
    os.makedirs(part_dir, exist_ok=True)
    n_parts = math.ceil(n_pages / PAGES_PER_PART)
    part_paths = []
    for i in range(n_parts):
        part_path = os.path.join(part_dir, f"{base}_part{i + 1}.pdf")
        if not os.path.exists(part_path):
            writer = PdfWriter()
            for page in reader.pages[i * PAGES_PER_PART: (i + 1) * PAGES_PER_PART]:
                writer.add_page(page)
            with open(part_path, "wb") as f:
                writer.write(f)
        part_paths.append(part_path)
    return part_paths


async def ingest_items(items, dataset_name):
    # reproducible as cognee uses LLM temperature = 0
    # rerun the ingestion cell to resume after interruption within this kernel
    for item in tqdm(items, desc=f"Ingesting {dataset_name}"):
        print(item["title"])
        if item["ingested"]:
            continue
        is_pdf = item.get("type") == "pdf"
        part_paths = split_pdf(item) if is_pdf else [item["local_path"]]
        for part_path in part_paths:
            await cognee.remember(
                part_path,
                dataset_name=dataset_name,
                node_set=item["sets"],
                self_improvement=False,  # per https://docs.cognee.ai/core-concepts/main-operations/remember#latency-and-cost
                data_per_batch=4,
            )
        item["ingested"] = True
        if is_pdf and len(part_paths) > 1:
            shutil.rmtree(os.path.dirname(part_paths[0]))  # remove temporary directory

In [ ]:
# populate Gammapy documentation, examples, reference papers, and the CTA whitepaper
await ingest_items(ingestion_items[GAMMAPY_CTA_HGPS_DATASET], dataset_name=GAMMAPY_CTA_HGPS_DATASET)

In [ ]:
# populate the remaining related papers in citation-count order
await ingest_items(ingestion_items[HESS_PAPERS_DATASET], dataset_name=HESS_PAPERS_DATASET)

In [ ]:
# show ingested files for both datasets
datasets = await cognee.datasets.list_datasets()
for dataset_name in DATASET_NAMES:
    dataset = next(d for d in datasets if d.name == dataset_name)
    ingested_items = await cognee.datasets.list_data(dataset.id)

    print(f"{dataset_name} — Count: {len(ingested_items)}")
    print("Ingested items:")
    for item in ingested_items:
        print(item.original_data_location)

In [ ]:
# query memory across both datasets
# question = "How many scientific publications mention Gammapy?"
question = "What is the easiest way to install Gammapy?"

results = await cognee.recall(
    query_text=question,
    datasets=DATASET_NAMES,
)

for result in results:
    print(result)

In [ ]:
from cognee.api.v1.visualize.visualize import visualize_graph

# visualize each dataset's knowledge graph
os.makedirs("../plot", exist_ok=True)
for dataset_name in DATASET_NAMES:
    graph_path = os.path.abspath(f"../plot/{dataset_name}_knowledge_graph.html")
    await visualize_graph(graph_path, dataset=dataset_name)

In [ ]:
# import time
#
# test_query = "What colormaps are used in gamma ray astronomy? Answer shortly!"
#
# # compare all search types with cache disabled
# for search_type in cognee.SearchType:
#     start = time.perf_counter()
#     try:
#         results = await cognee.recall(
#             query_text=test_query,
#             datasets=DATASET_NAMES,
#             query_type=search_type,
#         )
#         answer = results[0].text if results else "(no results)"
#     except Exception as e:
#         answer = f"ERROR: {e}"
#     elapsed = time.perf_counter() - start
#
#     print(f"QUERY_TYPE: {search_type.value}")
#     print(f"ANSWER: {answer}")
#     print(f"RETRIEVAL TIME: {elapsed:.2f}s")
#     print()
#
# # caching -> (self) improvement
# # GRAPH_COMPLETION is faster than RAG_COMPLETION
# # more concise retrieval and LLM prompt

In [ ]:
# export each dataset separately in portable format
for dataset_name in DATASET_NAMES:
    result = await cognee.export(
        dataset=dataset_name,
        format="cogx",
        destination=f"../data/cache/exported/{dataset_name}",
    )
    print(f"{dataset_name}: {result}")

In [ ]:
import psutil

started_pids = []
started_containers = []


def pid_callback(pid):
    # start_ui reports the MCP server as (pid, container_name)
    if isinstance(pid, tuple):
        _, container = pid
        print(f"Started container: {container}")
        started_containers.append(container)
        return
    print(f"Started process with PID: {pid}")
    started_pids.append(pid)


# start UI
server = cognee.start_ui(
    pid_callback,
    open_browser=False,
    auto_download=True,
    start_backend=True,
    start_mcp=True,
)

In [ ]:
def stop_processes(pids):
    for pid in pids:
        try:
            proc = psutil.Process(pid)
            procs = proc.children(recursive=True) + [proc]  # find children
            for p in procs:
                p.terminate()
            _, alive = psutil.wait_procs(procs, timeout=5)
            for p in alive:
                p.kill()  # force
            print(f"Stopped process {pid} (and {len(procs) - 1} children)")
        except Exception as e:
            print(f"Error stopping process {pid}: {e}")


def stop_containers(containers):
    for container in containers:
        subprocess.run(["docker", "stop", container], capture_output=True)  # also remove it
        print(f"Stopped container {container}")


# stop UI
stop_containers(started_containers)
stop_processes(started_pids)
# or server.terminate()